# Chapter 10 — Local Open WebUI + Scientific Service (v2026)

> **LangChain 1.x / 2026 refresh.** Dual-mode secrets, optional LangSmith tracing, pinned core deps where applicable, and a standardized footer (Limitations & safety + cleanup + exercises). **REFACTORED_LOCAL_OPENWEBUI_V2026**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IvanReznikov/LangChain4LifeSciencesHealthcare/blob/main/notebooks/CHDIR/FNAME)

## Learning objectives
- Understand Open WebUI Functions/Tools/Pipes as reviewable integration points
- Explore the local LifeScienceBench service endpoints
- Call the service locally and inspect a provenance manifest

> **Runtime / cost / data.** Offline demo of service endpoints with a local TestClient; no Docker required to run the cells.

## Environment setup

In [ ]:
import os

# Secrets are read from Colab Secrets if available, else from a local .env
try:
    from google.colab import userdata  # type: ignore

    def get_secret(name, default=""):
        return userdata.get(name) or default
except Exception:
    try:
        from dotenv import load_dotenv  # type: ignore

        load_dotenv()
    except Exception:
        pass

    def get_secret(name, default=""):
        return os.environ.get(name, default)

OPENAI_API_KEY = get_secret("LC4LS_OPENAI_API_KEY")
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# Optional LangSmith tracing (set LANGCHAIN_API_KEY to enable)
LANGSMITH_API_KEY = get_secret("LANGCHAIN_API_KEY", "")
LANGSMITH_PROJECT = "lc4lsh-chapter10-local-openwebui"
if LANGSMITH_API_KEY.startswith(("lsv2_", "ls__")):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_PROJECT", LANGSMITH_PROJECT)
    print("LangSmith tracing ON ->", LANGSMITH_PROJECT)
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith tracing OFF")

## Architecture

```
Open WebUI (UI)  -->  Pipes / Functions / Tools  -->  Local scientific service (FastAPI)
```

This repo ships a reference service at `lifesciencebench-v0.3.2/service/main.py` plus Open WebUI integrations in `lifesciencebench-v0.3.2/openwebui/` (`functions/`, `pipes/`, `actions/`). Everything is **reviewable source** — a key enterprise requirement.

### Reference Open WebUI assets in this repo

- `openwebui/functions/` — `evidence_first_filter.py`, `medical_research_filter.py`
- `openwebui/pipes/` — `lifesciencebench_{research,medical,chemistry,biology,physics,board}.py`
- `openwebui/actions/` — `export_run.py`
- `service/main.py` — FastAPI app: `/health`, `/projects`, `/runs`, `/corpus`

In [ ]:
# Minimal local stand-in for the service (mirrors service/main.py endpoints)
try:
    from fastapi import FastAPI
    from fastapi.testclient import TestClient
    HAVE_FASTAPI = True
except Exception:
    HAVE_FASTAPI = False
    print("fastapi not installed; showing conceptual calls only")

if HAVE_FASTAPI:
    app = FastAPI(title="Local Scientific Service (demo)")

    @app.get("/health")
    def health():
        return {"status": "ok", "release": "0.3.2", "network": "disabled"}

    @app.post("/projects")
    def project(payload: dict):
        return {"project_id": payload.get("project_id", "default"),
                "classification": payload.get("data_classification", "public")}

    @app.post("/runs")
    def run(payload: dict):
        q = payload.get("question", "")
        return {"question": q, "answer": "Metformin is first-line for T2DM.",
                "manifest": {"corpus_version": "abc123", "model": "local-mock"}}

    client = TestClient(app)
    print(client.get("/health").json())

In [ ]:
# Call the endpoints locally (no network)
if HAVE_FASTAPI:
    print("project:", client.post("/projects", json={"project_id": "demo", "data_classification": "internal"}).json())
    print("run:", client.post("/runs", json={"question": "First-line for T2DM?"}).json())

## Deploying with Open WebUI (reference)

```bash
# from lifesciencebench-v0.3.2/
docker compose up --build      # service + open-webui (see compose.yaml)
# then in Open WebUI: Admin > Functions / Pipes -> paste the files from openwebui/
```

- **Functions** transform/filter messages (e.g., `evidence_first_filter.py`).
- **Pipes** route a query to a backend pipeline (e.g., `lifesciencebench_medical.py`).
- **Tools/Actions** add buttons like `export_run.py`.
Because they are plain source, you can code-review each before enabling — essential for PHI/PII governance.

In [ ]:
# Inspect the provenance manifest returned by /runs
if HAVE_FASTAPI:
    res = client.post("/runs", json={"question": "First-line for T2DM?"}).json()
    print("manifest keys:", list(res["manifest"].keys()))
    assert res["manifest"]["corpus_version"], "provenance must record corpus version"
    print("Provenance manifest present and non-empty.")

## Limitations & safety

- **Enterprise / research-support only.** Human review is required before any production, clinical, or compliance decision.
- **Synthetic / de-identified data only.** Real PHI/PII requires governance and access controls.
- - The demo service is a local mock; the real service lives in `lifesciencebench-v0.3.2/service/` and runs offline by design.

In [ ]:
# Cleanup: drop references and free memory.
import gc

for _name in ["llm", "chain", "model", "agent", "app", "manifest", "report"]:
    globals().pop(_name, None)

gc.collect()
print("Cleanup complete.")

## Exercises

<details><summary>Q1. Why is a run manifest essential for reproducibility in an enterprise pipeline?</summary>
It captures corpus/index versions, prompts, and model/tool versions, so a past result can be audited, diffed, or re-run deterministically when models or data change.
</details>

<details><summary>Q2. Why treat guardrail / injection detections as drafts rather than automatic enforcement?</summary>
Detectors have false positives and negatives. In regulated settings, an error can block legitimate work or leak PHI, so a human confirms before enforcement.
</details>

<details><summary>Q3. Why compare frameworks by task and operations needs instead of popularity?</summary>
The best fit depends on state management, observability, deployment, and team skill — not download counts. A task-driven matrix makes the trade-offs explicit and defensible.
</details>

### Task A — Read `openwebui/functions/evidence_first_filter.py` and summarize what it filters in one paragraph.

### Task B — Add a `/corpus` endpoint to the demo app that accepts a document and returns a stored-id + parsed manifest.

### Task C — Add a data-classification guard that rejects `/runs` when `data_classification='restricted'` and route is external.

### Task D — Write a health-check script that pings `/health` and asserts `network == 'disabled'` before allowing queries.